# R16b · R16's grader corrected by the real grade shares (CPU, no GPU)

Task **R16b**, the follow-up to R16 in `Essential_task_delegation/Progress_Tracker.md` (go given on 19 Oct 2026). R16's notebook and results stay unchanged; this notebook only reads them.

**Why**
- R16's grader drew each grade equally often in training, so it acts as if a third of all buildings were destroyed. In the training masks, about 74% of buildings are intact and 8% destroyed.
- So it over-grades: of the 19,624 intact validation buildings, R16 graded 1,924 destroyed and 2,958 damaged. Destroyed precision was 0.35 for a recall of 0.89.

**The fix: a fixed rule, not tuned on validation**
- Bayes' rule with the right shares: each grade's probability is multiplied by its real share among the training buildings and divided by its share in training (a third), and the largest wins.
- The real shares come from the 5,798 training masks: 8-connected pieces of at least 20 px, the way R16 found its training buildings, without its cap of 12 intact pieces per image.
- This is the standard prior-shift correction (Saerens et al., 2002); R16's Cell 7 named it as the fix.

**What it does**
1. **Real shares:** rebuilds R16's pairs and split from `train_release.json` and counts the training buildings. The counts with R16's cap must equal R16's (42,814 / 41,781 / 18,121).
2. **R16 rebuilt:** recomputes R16's three maps from its per-building table (`instances_val.csv`) and E1's validation masks. The scores must equal `R16_summary.csv`.
3. **Corrected:** re-scores the three maps with the corrected grades (E-protocol, keep rule against R11 with TTA), and grades the stage-1 buildings that are mostly one true building.

**Inputs on this PC** (all already here):
- `train_release.json` and DisasterM3's masks in `dataset/DisasterM3 Dataset/DisasterM3_Instruct/masks/`;
- `Champion_Task/R16-results/R16/`: `instances_val.csv`, `R16_summary.csv`, `config.json`;
- `Champion_Task/E1-results/E1/gt/`: the 645 validation masks.

**Outputs in `Champion_Task/R16b-results/`:** `R16b_summary.csv`, `R16b_objects.csv`, `config.json`, `README.md`. About 9 minutes, mostly reading the training masks.

**Not done here:** R16's table of the grader on the true buildings needs the images and a GPU. The stage-1 buildings table stands in for it.

In [1]:
# ── Cell 1: Configuration and inputs (CPU only, no installs needed) ─────────
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image                # PIL and SciPy, as in D5b: this PC has no OpenCV
from scipy import ndimage
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# This PC: the repo root is the folder that holds train_release.json
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train_release.json").exists())
CT = REPO / "Essential_task_delegation" / "Champion_Task"
DATA_ROOT = REPO / "dataset" / "DisasterM3 Dataset" / "DisasterM3_Instruct"
MANIFEST_PATH = REPO / "train_release.json"
R16_DIR = CT / "R16-results" / "R16"
GT_DIR = CT / "E1-results" / "E1" / "gt"      # the 645 validation masks (0-3), saved by E1
OUT_DIR = CT / "R16b-results"

GRADES = ["Intact", "Damaged", "Destroyed"]
CLASS_NAMES = ["Background", *GRADES]
SOURCES = ("all", "xBD", "other")
TRAIN_SHARE = np.full(3, 1 / 3)     # R16's grader drew each grade equally often
R16_CONFIG = json.loads((R16_DIR / "config.json").read_text(encoding="utf-8"))
MIN_AREA = R16_CONFIG["min_area"]   # 20 px, R16's smallest building piece
KEEP_RULE = {"min_destroyed_gain": 0.013, "max_miou_drop": 0.005}     # R16's rule, against R11 with TTA

OUT_DIR.mkdir(parents=True, exist_ok=True)
t_start = time.time()
print(f"✓ R16b on this PC: {REPO}")
print(f"  R16 results: {R16_DIR}")
print(f"  E1 masks:    {GT_DIR} ({len(list(GT_DIR.glob('*.png')))} files)")
print(f"  Outputs:     {OUT_DIR}")

✓ R16b on this PC: D:\CSE499AB_project
  R16 results: D:\CSE499AB_project\Essential_task_delegation\Champion_Task\R16-results\R16
  E1 masks:    D:\CSE499AB_project\Essential_task_delegation\Champion_Task\E1-results\E1\gt (645 files)
  Outputs:     D:\CSE499AB_project\Essential_task_delegation\Champion_Task\R16b-results


## 1. The real grade shares, from the training masks
R16's pairs and split are rebuilt from `train_release.json` the way its Cells 3 and 4 build them. Only the masks are needed, read here with PIL. R16 kept every damaged and destroyed piece and capped intact pieces at 12 per image, so the capped counts must equal R16's.

In [2]:
# ── Cell 2: Training buildings per grade (R16's pairs and split, without the intact cap) ──
with open(MANIFEST_PATH, encoding="utf-8") as f:
    raw = json.load(f)
bda = [e for e in raw if e.get("task") == "Referring Expression Segmentation"
       and e.get("cls_description") == "Building Damage Assessment"]

def resolve_mask(rel_path):
    """The mask candidates of the D4 notebooks' resolve_path (this PC holds the masks, not the images)."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    for c in (DATA_ROOT / rel_path, DATA_ROOT / "masks" / rel_path, DATA_ROOT / "masks" / "masks" / Path(rel_path).name):
        if c.exists():
            return c
    return None

FOLDER_TO_CLASS = {"train_building_intact_mask": 1, "train_building_damaged_mask": 2,
                   "train_building_destroyed_mask": 3}
by_image = defaultdict(dict)        # after-image -> {mask folder: path}, in manifest order as in R16
for e in bda:
    if e.get("image_type") != "Optical":
        continue
    rel = e.get("ground_truth", "")
    path = resolve_mask(rel)
    if path:
        by_image[e.get("post_image_path", "")][Path(rel.replace("\\", "/")).parent.name] = path

names = list(by_image)              # R16 built one pair per image, in this order
assert len(names) == R16_CONFIG["train_images"] + R16_CONFIG["val_images"], f"{len(names):,} images; R16 had 6,443"
train_names, val_names = train_test_split(names, test_size=0.1, random_state=42)

def combined_mask(masks):
    """The D4 notebooks' 4-class mask: later classes overwrite earlier ones."""
    out = None
    for folder, cls in FOLDER_TO_CLASS.items():
        if folder in masks:
            m = np.array(Image.open(masks[folder]).convert("L"))
            out = np.zeros_like(m, np.uint8) if out is None else out
            out[m > 0] = cls
    return out

EIGHT = np.ones((3, 3), bool)       # 8-connected, as connectivity=8 in R16
pieces = np.zeros(3, np.int64)      # every piece of at least MIN_AREA px
capped = np.zeros(3, np.int64)      # with R16's cap on intact pieces
for n in tqdm(train_names, desc="Training masks"):
    gt = combined_mask(by_image[n])
    for cls in (1, 2, 3):
        labels, _ = ndimage.label(gt == cls, structure=EIGHT)
        k = int((np.bincount(labels.ravel())[1:] >= MIN_AREA).sum())
        pieces[cls - 1] += k
        capped[cls - 1] += min(k, R16_CONFIG["max_intact_per_image"]) if cls == 1 else k
assert capped.tolist() == [R16_CONFIG["train_buildings"][g] for g in GRADES], \
    f"{capped.tolist()} differs from R16's training buildings {R16_CONFIG['train_buildings']}"
REAL_SHARE = pieces / pieces.sum()
print(f"✓ Training buildings with R16's cap: {capped.tolist()}, the same as R16's")
print("  Without the cap (the real shares): "
      + ", ".join(f"{g} {c:,} ({s:.1%})" for g, c, s in zip(GRADES, pieces, REAL_SHARE)))

Training masks:   0%|          | 0/5798 [00:00<?, ?it/s]

✓ Training buildings with R16's cap: [42814, 41781, 18121], the same as R16's
  Without the cap (the real shares): Intact 167,706 (73.7%), Damaged 41,781 (18.4%), Destroyed 18,121 (8.0%)


## 2. R16's three maps, rebuilt from its per-building table
Every pixel belongs to at most one stage-1 building, and the pixels outside them are background in all three R16 maps. So each map's confusion matrix is every building's true pixel counts added to the column of its class, plus the remaining true pixels in the background column. The rebuilt scores must equal `R16_summary.csv`.

In [3]:
# ── Cell 3: Validation masks, and R16's three maps rebuilt from instances_val.csv ──
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")   # R16's rule: xBD tiles, everything else is "other"
gt_files = sorted(GT_DIR.glob("*.png"))
assert {p.name for p in gt_files} == {Path(n.replace("\\", "/")).name for n in val_names}, \
    "E1's masks aren't R16's 645 validation images"

totals = {s: np.zeros(4, np.int64) for s in SOURCES}      # true pixels per class
for p in tqdm(gt_files, desc="Validation masks"):
    counts = np.bincount(np.array(Image.open(p).convert("L")).ravel(), minlength=4)
    assert len(counts) == 4, f"{p.name} has values above 3"
    totals["all"] += counts
    totals["xBD" if XBD_NAME.match(p.name) else "other"] += counts

inst = pd.read_csv(R16_DIR / "instances_val.csv")
P_COLS = [f"p_{g.lower()}" for g in GRADES]
GT_COLS = [f"gt_{c.lower()}_px" for c in CLASS_NAMES]
R11_COLS = [f"r11_{g.lower()}_px" for g in GRADES]

def r16_maps(rows, probs):
    """Each building's class (0-3) in R16's three maps, as R16's Cell 9 paints them."""
    by_grader = probs.argmax(1) + 1
    r11 = rows[R11_COLS].to_numpy(np.float64)
    r11_sum = r11.sum(1, keepdims=True)
    shares = np.where(r11_sum > 0, r11 / np.maximum(r11_sum, 1), probs)   # no R11 building pixel: grader only
    return {"R16_sam3": np.where(rows["stage1"].to_numpy() == "sam3", by_grader, 0),   # R11's pieces left out
            "R16_sam3+r11": by_grader,
            "R16_vote": (0.5 * (probs + shares)).argmax(1) + 1}

def confusion(rows, building_class, total):
    """4×4 pixel counts of one map (rows truth, columns prediction)."""
    gt_px = rows[GT_COLS].to_numpy(np.int64)
    cm = np.zeros((4, 4), np.int64)
    for k in range(4):
        cm[:, k] += gt_px[building_class == k].sum(0)
    cm[:, 0] += total - gt_px.sum(0)          # outside every stage-1 building: background
    return cm

def metrics(cm):
    """R16's dataset-level metrics (E1's formulas)."""
    cm = cm.astype(np.float64)
    eps = 1e-8
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    building_tp = cm[1:, 1:].sum()
    return {"iou": tp / (tp + fp + fn + eps), "precision": tp / (tp + fp + eps), "recall": tp / (tp + fn + eps),
            "building_iou": building_tp / (building_tp + cm[0, 1:].sum() + cm[1:, 0].sum() + eps)}

def score(probs, label):
    """R16_summary.csv's rows for the three maps, overall and per source."""
    out = []
    for source in SOURCES:
        sel = np.ones(len(inst), bool) if source == "all" else (inst["source"] == source).to_numpy()
        for variant, building_class in r16_maps(inst[sel], probs[sel]).items():
            m = metrics(confusion(inst[sel], building_class, totals[source]))
            out.append({"grades": label, "variant": variant, "source": source, "miou": round(float(m["iou"].mean()), 4),
                        **{f"{c.lower()}_iou": round(float(m["iou"][i]), 4) for i, c in enumerate(CLASS_NAMES)},
                        "building_iou": round(float(m["building_iou"]), 4),
                        "destroyed_precision": round(float(m["precision"][3]), 4),
                        "destroyed_recall": round(float(m["recall"][3]), 4)})
    return pd.DataFrame(out)

P = inst[P_COLS].to_numpy(np.float64)
original = score(P, "R16 (equal shares)")
r16 = pd.read_csv(R16_DIR / "R16_summary.csv")
check = original.merge(r16, on=["variant", "source"], suffixes=("", "_r16"))
worst = max(float((check[c] - check[f"{c}_r16"]).abs().max()) for c in r16.columns if c not in ("variant", "source"))
assert len(check) == 9 and worst < 1e-4, f"the rebuilt scores differ from R16_summary.csv by up to {worst}"
print(f"✓ {len(inst):,} stage-1 buildings on {inst['image'].nunique()} images; "
      f"R16's 9 rows rebuilt exactly (largest difference {worst:.4f})")

Validation masks:   0%|          | 0/645 [00:00<?, ?it/s]

✓ 57,991 stage-1 buildings on 631 images; R16's 9 rows rebuilt exactly (largest difference 0.0000)


## 3. The correction, and the new scores
`p'(grade) ∝ p(grade) × real share / training share`, renormalised. Nothing is fitted on the validation images: the shares come from Cell 2 and from R16's sampler. The keep rule is R16's, against R11 with TTA. The last table grades the stage-1 buildings that have at least half their pixels on true buildings, each by its main true grade.

In [4]:
# ── Cell 4: Corrected grades, the keep rule, and the stage-1 buildings' grades ──
Pc = P * (REAL_SHARE / TRAIN_SHARE)
Pc /= Pc.sum(1, keepdims=True)
corrected = score(Pc, "R16b (real shares)")
summary = pd.concat([original, corrected], ignore_index=True)
print(summary[summary["source"] == "all"].drop(columns="source").to_string(index=False))

r11 = r16[(r16["variant"] == "R11_tta") & (r16["source"] == "all")].iloc[0]
verdicts = {}
print(f"\nKeep rule against R11 TTA (mIoU {r11['miou']:.4f}, Destroyed IoU {r11['destroyed_iou']:.4f}):")
for _, row in corrected[corrected["source"] == "all"].iterrows():
    d_miou, d_destroyed = row["miou"] - r11["miou"], row["destroyed_iou"] - r11["destroyed_iou"]
    verdicts[row["variant"]] = bool(d_destroyed > KEEP_RULE["min_destroyed_gain"]
                                    and d_miou >= -KEEP_RULE["max_miou_drop"])
    print(f"  {row['variant']:>13}: mIoU {row['miou']:.4f} ({d_miou:+.4f}) | Destroyed IoU {row['destroyed_iou']:.4f} "
          f"({d_destroyed:+.4f}) → {'KEEP' if verdicts[row['variant']] else 'not kept'}")

by_source = pd.concat([r16[r16["variant"] == "R11_tta"].assign(grades="reference"), summary])
by_source = by_source[by_source["source"] != "all"][["grades", "variant", "source", "miou", "destroyed_iou",
                                                     "destroyed_precision", "destroyed_recall"]]
print("\nBy source (xBD 560 images, other 85):")
print(by_source.to_string(index=False))

# Stage-1 buildings that are mostly one true building, graded before and after
gt_grades = inst[[f"gt_{g.lower()}_px" for g in GRADES]].to_numpy(np.int64)
real_building = gt_grades.sum(1) >= 0.5 * inst["area_px"].to_numpy()
truth = gt_grades.argmax(1)
object_rows, macro_f1 = [], {}
for label, probs in (("R16 (equal shares)", P), ("R16b (real shares)", Pc)):
    cm = np.zeros((3, 3), np.int64)
    np.add.at(cm, (truth[real_building], probs[real_building].argmax(1)), 1)
    precision = np.diag(cm) / np.maximum(cm.sum(0), 1)
    recall = np.diag(cm) / np.maximum(cm.sum(1), 1)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-8)
    macro_f1[label] = round(float(f1.mean()), 4)
    object_rows += [{"grades": label, "true_grade": g, "buildings": int(cm[i].sum()),
                     "precision": round(float(precision[i]), 4), "recall": round(float(recall[i]), 4),
                     "f1": round(float(f1[i]), 4), **{f"graded_{h.lower()}": int(cm[i, j]) for j, h in enumerate(GRADES)}}
                    for i, g in enumerate(GRADES)]
objects = pd.DataFrame(object_rows)
grade_counts = {label: dict(zip(GRADES, np.bincount(probs.argmax(1), minlength=3).tolist()))
                for label, probs in (("R16 (equal shares)", P), ("R16b (real shares)", Pc))}
print(f"\nStage-1 buildings that are mostly one true building: {int(real_building.sum()):,} of {len(inst):,}")
print(objects.to_string(index=False))
print(f"Macro F1: {macro_f1}")
print(f"All {len(inst):,} stage-1 buildings by grade: {grade_counts}")

            grades      variant   miou  background_iou  intact_iou  damaged_iou  destroyed_iou  building_iou  destroyed_precision  destroyed_recall
R16 (equal shares)     R16_sam3 0.4711          0.9685      0.4713       0.2772         0.1672        0.6304               0.1902            0.5813
R16 (equal shares) R16_sam3+r11 0.4941          0.9752      0.5690       0.2800         0.1522        0.7426               0.1600            0.7576
R16 (equal shares)     R16_vote 0.5796          0.9752      0.6846       0.3975         0.2609        0.7426               0.4142            0.4135
R16b (real shares)     R16_sam3 0.5150          0.9685      0.5450       0.3186         0.2279        0.6304               0.2809            0.5474
R16b (real shares) R16_sam3+r11 0.5407          0.9752      0.6441       0.3357         0.2079        0.7426               0.2270            0.7125
R16b (real shares)     R16_vote 0.5846          0.9752      0.6864       0.4033         0.2737        0.7426    

       R16_sam3: mIoU 0.5150 (-0.0725) | Destroyed IoU 0.2279 (-0.0686) → not kept
   R16_sam3+r11: mIoU 0.5407 (-0.0468) | Destroyed IoU 0.2079 (-0.0886) → not kept
       R16_vote: mIoU 0.5846 (-0.0029) | Destroyed IoU 0.2737 (-0.0228) → not kept

By source (xBD 560 images, other 85):
            grades      variant source   miou  destroyed_iou  destroyed_precision  destroyed_recall
         reference      R11_tta    xBD 0.6462         0.4562               0.6348            0.6186
         reference      R11_tta  other 0.4316         0.0707               0.2516            0.0896
R16 (equal shares)     R16_sam3    xBD 0.5031         0.2321               0.2919            0.5310
R16 (equal shares) R16_sam3+r11    xBD 0.5072         0.1800               0.1929            0.7292
R16 (equal shares)     R16_vote    xBD 0.6237         0.3770               0.4706            0.6545
R16 (equal shares)     R16_sam3  other 0.3901         0.1284               0.1380            0.6478
R16 (equal s

In [5]:
# ── Cell 5: Output files ───────────────────────────────────────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

summary.to_csv(OUT_DIR / "R16b_summary.csv", index=False)
objects.to_csv(OUT_DIR / "R16b_objects.csv", index=False)

overall = pd.concat([r16[r16["variant"].str.startswith("R11")].assign(grades="reference"), summary])
overall = overall[overall["source"] == "all"][["grades", "variant", "miou", "intact_iou", "damaged_iou", "destroyed_iou",
                                               "building_iou", "destroyed_precision", "destroyed_recall"]]
shares = ", ".join(f"{g} {s:.1%}" for g, s in zip(GRADES, REAL_SHARE))
config = {"task": "R16b", "rule": "p' ∝ p × real share / training share",
          "training_share": TRAIN_SHARE.round(4).tolist(), "real_share": REAL_SHARE.round(4).tolist(),
          "training_buildings_uncapped": dict(zip(GRADES, pieces.tolist())),
          "stage1_buildings": len(inst), "mostly_one_true_building": int(real_building.sum()),
          "stage1_macro_f1": macro_f1, "grade_counts": grade_counts, "keep": verdicts,
          "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False), encoding="utf-8")

(OUT_DIR / "README.md").write_text("\n".join([
    "# R16b results: R16's grader corrected by the real grade shares", "",
    "R16's grader drew each grade equally often in training. Here each grade's probability is multiplied by its real "
    f"share among the training buildings ({shares}) and divided by a third, and the largest wins. The shares come "
    "from the 5,798 training masks, so nothing is fitted on the validation images. R16's own scores were first "
    "rebuilt exactly from its per-building table.", "",
    "## Pixel scores (E-protocol, 645 validation images)", "", as_markdown(overall), "",
    "## Keep rule against R11 TTA", "",
    *[f"- {v}: {'KEEP' if k else 'not kept'}" for v, k in verdicts.items()], "",
    "## By source (xBD 560 images, other 85)", "", as_markdown(by_source), "",
    "## Stage-1 buildings that are mostly one true building", "",
    f"{int(real_building.sum()):,} of the {len(inst):,} stage-1 buildings have at least half their pixels on true "
    "buildings; each is compared with its main true grade. R16's table of the grader on the true buildings needs the "
    "images and a GPU, so this one stands in for it.", "", as_markdown(objects), "",
    f"Macro F1: {macro_f1}.", "",
    "## Files", "",
    "- `R16b_summary.csv`: the three R16 maps with R16's grades and with the corrected ones, overall and per source",
    "- `R16b_objects.csv`: the stage-1 buildings' grades against their main true grade",
    "- `config.json`: the shares, the building counts and the keep verdicts",
]), encoding="utf-8")
print(f"✓ Wrote R16b_summary.csv, R16b_objects.csv, config.json and README.md to {OUT_DIR} "
      f"({(time.time() - t_start) / 60:.1f} min)")

✓ Wrote R16b_summary.csv, R16b_objects.csv, config.json and README.md to D:\CSE499AB_project\Essential_task_delegation\Champion_Task\R16b-results (7.3 min)
